# Notebook 08 — Fault-Family-Held-Out Threshold Calibration & Evaluation

**PINN EV Battery / 100-cell digital twin.** This notebook evaluates the signal-statistics detector from Notebook 07 using family-isolated splits. It does not retrain the PINN or build the frontend/backend.

Research safeguards:
- Calibrate the threshold using training families only.
- Check family leakage in existing splits; rebuild deterministic family-level splits if needed.
- Report specificity only when healthy test scenarios exist.
- Do not claim early-warning lead time: fault-onset timestamps are unavailable.
- Do not claim real-world battery-safety validation from simulation results.


In [1]:
# Cell 1 — Google Drive and setup
from pathlib import Path
import json, os
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as exc:
    raise RuntimeError("Run this notebook in Google Colab.") from exc

SEED = 20261014
N_CELLS = 100
PROJECT_ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
DATA_DIR = PROJECT_ROOT / "data" / "v1"
RESULTS_DIR = PROJECT_ROOT / "results_v1_localization"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)


Mounted at /content/drive
Project: /content/drive/MyDrive/PINN_EV_Battery


In [2]:
# Cell 2 — Find exact inputs; stop if ambiguous
def locate_one(preferred, filename, root):
    existing = [p for p in preferred if p.is_file()]
    if len(existing) == 1:
        return existing[0]
    if len(existing) > 1:
        raise FileNotFoundError(f"Multiple candidate files for {filename}: {existing}")
    matches = [p for p in root.rglob(filename) if p.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; found {len(matches)}. "
            f"Matches: {[str(p) for p in matches[:20]]}"
        )
    return matches[0]

SCENARIO_CSV = locate_one(
    [DATA_DIR / "scenario_instances.csv", PROJECT_ROOT / "scenario_instances.csv"],
    "scenario_instances.csv", PROJECT_ROOT
)
SCORES_CSV = locate_one(
    [RESULTS_DIR / "07_signal_localization_diagnostic.csv"],
    "07_signal_localization_diagnostic.csv", PROJECT_ROOT
)
print("Scenario metadata:", SCENARIO_CSV)
print("Signal scores:", SCORES_CSV)


Scenario metadata: /content/drive/MyDrive/PINN_EV_Battery/data/v1/scenario_instances.csv
Signal scores: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_signal_localization_diagnostic.csv


In [3]:
# Cell 3 — Load and validate schemas
metadata = pd.read_csv(SCENARIO_CSV, low_memory=False)
scores = pd.read_csv(SCORES_CSV, low_memory=False)

def col(df, choices, label):
    for c in choices:
        if c in df.columns:
            return c
    raise ValueError(f"Missing {label}; expected one of {choices}. Actual columns: {list(df.columns)}")

ID = col(metadata, ["scenario_instance_id"], "scenario_instance_id")
SCENARIO = col(metadata, ["scenario_id"], "scenario_id")
FAULTS = col(metadata, ["fault_cells_json"], "fault_cells_json")
BASE = "base_scenario_instance_id" if "base_scenario_instance_id" in metadata.columns else None
SPLIT = "split" if "split" in metadata.columns else ("data_split" if "data_split" in metadata.columns else None)
SID = col(scores, ["scenario_instance_id"], "score scenario_instance_id")
CELL = col(scores, ["cell_id"], "cell_id")
SCORE = col(scores, ["signal_anomaly_score"], "signal_anomaly_score")

metadata[ID] = metadata[ID].astype(str).str.strip()
metadata[SCENARIO] = metadata[SCENARIO].astype(str).str.strip()
scores[SID] = scores[SID].astype(str).str.strip()
scores[CELL] = pd.to_numeric(scores[CELL], errors="coerce")
scores[SCORE] = pd.to_numeric(scores[SCORE], errors="coerce")

if metadata[ID].duplicated().any():
    raise ValueError("Duplicate scenario_instance_id values in scenario metadata.")
if scores[[CELL, SCORE]].isna().any().any():
    raise ValueError("Cell IDs or anomaly scores contain missing/non-numeric values.")
if not np.isfinite(scores[SCORE].to_numpy(float)).all():
    raise ValueError("Non-finite anomaly scores detected.")
scores[CELL] = scores[CELL].astype(int)
print("Metadata rows:", len(metadata), "| score rows:", len(scores))
print("Score columns:", list(scores.columns))


Metadata rows: 28 | score rows: 2800
Score columns: ['scenario_instance_id', 'scenario_id', 'cell_id', 'signal_anomaly_score', 'signal_rank']


In [4]:
# Cell 4 — Parse labels and construct canonical fault-family IDs
def parse_faults(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return set()
    if isinstance(value, (list, tuple, set, np.ndarray)):
        items = list(value)
    else:
        raw = str(value).strip()
        if raw.lower() in {"", "nan", "none", "null"}:
            return set()
        items = json.loads(raw)
    result = set()
    for x in (items or []):
        if x is None:
            continue
        cell = int(x)
        if not 1 <= cell <= N_CELLS:
            raise ValueError(f"Invalid fault cell ID {cell}; expected 1..100.")
        result.add(cell)
    return result

metadata["fault_cells"] = metadata[FAULTS].apply(parse_faults)
metadata["has_fault"] = metadata["fault_cells"].apply(bool)

# Default family: scenario family + fault signature. A base pointer overrides
# this and is mapped to the canonical family of the referenced base scenario.
def default_family(row):
    signature = ",".join(map(str, sorted(row["fault_cells"])))
    return f"SCENARIO::{row[SCENARIO]}::FAULTS::{signature}"

metadata["family_key"] = metadata.apply(default_family, axis=1)
id_to_family = dict(zip(metadata[ID], metadata["family_key"]))
if BASE is not None:
    for idx, value in metadata[BASE].items():
        if pd.isna(value) or str(value).strip().lower() in {"", "nan", "none", "null"}:
            continue
        base_id = str(value).strip()
        metadata.at[idx, "family_key"] = id_to_family.get(base_id, f"BASE::{base_id}")

missing_meta = sorted(set(scores[SID]) - set(metadata[ID]))
if missing_meta:
    raise ValueError(f"Scores refer to scenarios absent from metadata: {missing_meta[:20]}")
if scores.duplicated([SID, CELL]).any():
    raise ValueError("Duplicate rows for the same scenario/cell; expected one summary score per cell.")
bad_ids = sorted(set(scores[CELL]) - set(range(1, N_CELLS + 1)))
if bad_ids:
    raise ValueError(f"Cell IDs outside 1..100: {bad_ids[:20]}")

coverage = scores.groupby(SID)[CELL].agg(lambda x: sorted(set(x)))
bad_coverage = {sid: len(ids) for sid, ids in coverage.items()
                if ids != list(range(1, N_CELLS + 1))}
if bad_coverage:
    raise ValueError(f"Each scored scenario must contain exactly cells 1..100. Bad scenarios: {bad_coverage}")

print("Unique families:", metadata["family_key"].nunique())
print("Healthy scenarios:", int((~metadata["has_fault"]).sum()))
print("Fault scenarios:", int(metadata["has_fault"].sum()))
print("100-cell coverage: PASS")


Unique families: 13
Healthy scenarios: 3
Fault scenarios: 25
100-cell coverage: PASS


In [5]:
# Cell 5 — Audit current split for family leakage
existing_valid = False
split_audit = {}
if SPLIT is not None:
    metadata["_old_split"] = metadata[SPLIT].astype(str).str.strip().str.lower()
    observed = set(metadata["_old_split"])
    allowed = {"train", "val", "validation", "test"}
    if observed.issubset(allowed):
        metadata["_old_split"] = metadata["_old_split"].replace({"validation": "val"})
        family_counts = metadata.groupby("family_key")["_old_split"].nunique()
        overlap = family_counts[family_counts > 1].index.tolist()
        old_test = metadata[metadata["_old_split"] == "test"]
        test_healthy = bool((~old_test["has_fault"]).any())
        test_faulty = bool(old_test["has_fault"].any())
        existing_valid = (
            not overlap and test_healthy and test_faulty
            and {"train", "test"}.issubset(set(metadata["_old_split"]))
        )
        split_audit = {
            "family_overlap_count": len(overlap),
            "overlapping_families": overlap,
            "existing_test_has_healthy": test_healthy,
            "existing_test_has_fault": test_faulty,
        }
    else:
        split_audit["invalid_split_values"] = sorted(observed - allowed)
print(json.dumps(split_audit, indent=2, default=str))
print("Retain existing split:", existing_valid)


{
  "family_overlap_count": 0,
  "overlapping_families": [],
  "existing_test_has_healthy": false,
  "existing_test_has_fault": true
}
Retain existing split: False


In [6]:
# Cell 6 — Use existing split only if family-isolated; otherwise split whole families
rng = np.random.default_rng(SEED)
if existing_valid:
    metadata["evaluation_split"] = metadata["_old_split"]
    split_source = "existing split retained after family-isolation checks"
else:
    family_table = metadata.groupby("family_key", as_index=False).agg(
        has_fault=("has_fault", "max"),
        scenario_count=(ID, "nunique")
    )
    assignment = {}
    # Allocate families separately by healthy/fault family class to avoid
    # accidentally putting all healthy or all fault families in one split.
    for class_value in [False, True]:
        families = family_table.loc[family_table["has_fault"] == class_value, "family_key"].tolist()
        rng.shuffle(families)
        n = len(families)
        if n == 0:
            continue
        if n == 1:
            allocations = ["train"]
        elif n == 2:
            allocations = ["train", "test"]
        else:
            n_train = max(1, int(round(0.60 * n)))
            n_val = max(1, int(round(0.20 * n)))
            n_test = n - n_train - n_val
            if n_test < 1:
                n_test = 1
                if n_train > 1:
                    n_train -= 1
                else:
                    n_val -= 1
            allocations = ["train"] * n_train + ["val"] * n_val + ["test"] * n_test
            allocations = allocations[:n]
            while len(allocations) < n:
                allocations.append("train")
            rng.shuffle(allocations)
        assignment.update(dict(zip(families, allocations)))
    metadata["evaluation_split"] = metadata["family_key"].map(assignment)
    if metadata["evaluation_split"].isna().any():
        raise RuntimeError("One or more families were not assigned a split.")
    split_source = "rebuilt deterministic family-level split"

family_nsplit = metadata.groupby("family_key")["evaluation_split"].nunique()
if (family_nsplit > 1).any():
    raise RuntimeError("Family leakage remains after split assignment.")

split_overview = metadata.groupby("evaluation_split").agg(
    scenarios=(ID, "nunique"),
    families=("family_key", "nunique"),
    healthy_scenarios=("has_fault", lambda x: int((~x).sum())),
    fault_scenarios=("has_fault", "sum")
).reset_index()
print("Split source:", split_source)
display(split_overview)
print("Family isolation: PASS")


Split source: rebuilt deterministic family-level split


,evaluation_split,scenarios,families,healthy_scenarios,fault_scenarios
0,test,8,3,1,7
1,train,17,7,1,16
2,val,3,3,1,2


Family isolation: PASS


In [7]:
# Cell 7 — Join labels after anomaly scores are loaded; prepare evaluation table
meta_join = metadata[[ID, SCENARIO, "family_key", "evaluation_split", "fault_cells", "has_fault"]].rename(
    columns={ID: "scenario_instance_id"}
)
cell_eval = scores.rename(columns={SID: "scenario_instance_id", CELL: "cell_id", SCORE: "signal_anomaly_score"}).copy()
cell_eval = cell_eval.merge(meta_join, on="scenario_instance_id", how="left", validate="many_to_one")
if cell_eval["evaluation_split"].isna().any():
    raise RuntimeError("A score row has no assigned split.")
cell_eval["is_fault"] = [
    int(int(cid) in fault_set)
    for cid, fault_set in zip(cell_eval["cell_id"], cell_eval["fault_cells"])
]
print("Evaluation rows:", f"{len(cell_eval):,}")
display(cell_eval.groupby("evaluation_split").size().rename("scored_cells").to_frame())


Evaluation rows: 2,800


,scored_cells
evaluation_split,
test,800
train,1700
val,300


In [8]:
# Cell 8 — Calibrate threshold using TRAINING families only
train_df = cell_eval[cell_eval["evaluation_split"] == "train"].copy()
healthy_scores = train_df.loc[train_df["is_fault"] == 0, "signal_anomaly_score"]
fault_scores = train_df.loc[train_df["is_fault"] == 1, "signal_anomaly_score"]
if healthy_scores.empty or fault_scores.empty:
    raise RuntimeError(
        "Training split must contain both healthy and faulty cells to calibrate the requested threshold. "
        "There are not enough independent families for this split; expand/revise scenario design."
    )

max_healthy_train = float(healthy_scores.max())
min_fault_train = float(fault_scores.min())
threshold = (max_healthy_train + min_fault_train) / 2.0
separation_gap = min_fault_train - max_healthy_train
train_separated = separation_gap > 0

threshold_report = {
    "threshold": threshold,
    "rule": "midpoint(max training healthy score, min training faulty score)",
    "max_training_healthy_score": max_healthy_train,
    "min_training_fault_score": min_fault_train,
    "training_separation_gap": separation_gap,
    "training_perfect_separation": bool(train_separated),
    "threshold_calibrated_using_training_families_only": True,
    "split_source": split_source,
    "seed": SEED
}
print(json.dumps(threshold_report, indent=2))
if not train_separated:
    print("WARNING: Training score distributions overlap; midpoint threshold is not a perfect-separation threshold.")

# Freeze threshold before evaluating validation/test.
cell_eval["predicted_fault"] = cell_eval["signal_anomaly_score"] >= threshold


{
  "threshold": 4.399034460052955,
  "rule": "midpoint(max training healthy score, min training faulty score)",
  "max_training_healthy_score": 1.655211777248763,
  "min_training_fault_score": 7.142857142857148,
  "training_separation_gap": 5.487645365608385,
  "training_perfect_separation": true,
  "threshold_calibrated_using_training_families_only": true,
  "split_source": "rebuilt deterministic family-level split",
  "seed": 20261014
}


In [9]:
# Cell 9 — Cell-level confusion metrics by split
def div(a, b):
    return float(a / b) if b else float("nan")

def metrics_for(df, split_name):
    y = df["is_fault"].astype(bool).to_numpy()
    pred = df["predicted_fault"].astype(bool).to_numpy()
    tp = int(np.sum(pred & y))
    fp = int(np.sum(pred & ~y))
    fn = int(np.sum(~pred & y))
    tn = int(np.sum(~pred & ~y))
    precision = div(tp, tp + fp)
    recall = div(tp, tp + fn)
    specificity = div(tn, tn + fp)
    f1 = div(2 * precision * recall, precision + recall) if np.isfinite(precision) and np.isfinite(recall) and precision + recall else float("nan")
    return {
        "split": split_name,
        "families": int(df["family_key"].nunique()),
        "scenarios": int(df["scenario_instance_id"].nunique()),
        "healthy_scenarios": int(df.loc[~df["has_fault"], "scenario_instance_id"].nunique()),
        "fault_scenarios": int(df.loc[df["has_fault"], "scenario_instance_id"].nunique()),
        "cells": int(len(df)),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": precision, "recall": recall, "specificity": specificity, "F1": f1
    }

metric_rows = []
for name in ["train", "val", "test"]:
    part = cell_eval[cell_eval["evaluation_split"] == name]
    if not part.empty:
        metric_rows.append(metrics_for(part, name))
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
test_metric = metrics_df[metrics_df["split"] == "test"]
if test_metric.empty:
    print("WARNING: No test scenarios; held-out evaluation is impossible.")
else:
    tm = test_metric.iloc[0]
    if int(tm["healthy_scenarios"]) == 0:
        print("WARNING: No healthy test scenarios; test specificity is not established.")
    if int(tm["fault_scenarios"]) == 0:
        print("WARNING: No faulty test scenarios; held-out fault recall is not established.")


,split,families,scenarios,healthy_scenarios,fault_scenarios,cells,TP,FP,FN,TN,precision,recall,specificity,F1
0,train,7,17,1,16,1700,22,0,0,1678,1.0,1.0,1.0,1.0
1,val,3,3,1,2,300,2,0,0,298,1.0,1.0,1.0,1.0
2,test,3,8,1,7,800,7,0,0,793,1.0,1.0,1.0,1.0


In [11]:

# Cell 10 — Per-scenario results: actual vs predicted faulty cells

# Resolve the scenario_id collision created by Cell 7's merge.
# Prefer scenario_id_y (from metadata), then scenario_id_x (from scores).
if "scenario_id" not in cell_eval.columns:
    if "scenario_id_y" in cell_eval.columns:
        cell_eval["scenario_id"] = cell_eval["scenario_id_y"]
    elif "scenario_id_x" in cell_eval.columns:
        cell_eval["scenario_id"] = cell_eval["scenario_id_x"]
    else:
        # Last-resort recovery directly from the metadata dataframe.
        scenario_lookup = metadata[[ID, SCENARIO]].copy()
        scenario_lookup[ID] = scenario_lookup[ID].astype(str).str.strip()
        scenario_lookup = scenario_lookup.rename(
            columns={ID: "scenario_instance_id", SCENARIO: "scenario_id"}
        )

        cell_eval["scenario_instance_id"] = (
            cell_eval["scenario_instance_id"].astype(str).str.strip()
        )
        cell_eval = cell_eval.merge(
            scenario_lookup,
            on="scenario_instance_id",
            how="left",
            validate="many_to_one"
        )

if cell_eval["scenario_id"].isna().any():
    raise ValueError("Some rows are missing scenario_id after recovery.")

scenario_rows = []

for sid, group in cell_eval.groupby("scenario_instance_id", sort=True):
    truth = set(
        group.loc[group["is_fault"] == 1, "cell_id"].astype(int)
    )
    predicted = set(
        group.loc[group["predicted_fault"], "cell_id"].astype(int)
    )

    fp_cells = sorted(predicted - truth)
    fn_cells = sorted(truth - predicted)

    scenario_rows.append({
        "scenario_instance_id": sid,
        "scenario_id": str(group["scenario_id"].iloc[0]),
        "family_key": group["family_key"].iloc[0],
        "split": group["evaluation_split"].iloc[0],
        "actual_fault_cells": json.dumps(sorted(truth)),
        "predicted_fault_cells": json.dumps(sorted(predicted)),
        "false_positive_cells": json.dumps(fp_cells),
        "false_negative_cells": json.dumps(fn_cells),
        "actual_fault_count": len(truth),
        "predicted_fault_count": len(predicted),
        "false_positive_count": len(fp_cells),
        "false_negative_count": len(fn_cells),
        "exact_cell_set_match": bool(truth == predicted),
        "healthy_false_alarm": bool(not truth and bool(predicted))
    })

scenario_df = pd.DataFrame(scenario_rows)

print("Scenario rows generated:", len(scenario_df))
print("Columns:", list(scenario_df.columns))
display(
    scenario_df[scenario_df["split"] == "test"]
    .reset_index(drop=True)
)


Scenario rows generated: 28
Columns: ['scenario_instance_id', 'scenario_id', 'family_key', 'split', 'actual_fault_cells', 'predicted_fault_cells', 'false_positive_cells', 'false_negative_cells', 'actual_fault_count', 'predicted_fault_count', 'false_positive_count', 'false_negative_count', 'exact_cell_set_match', 'healthy_false_alarm']


,scenario_instance_id,scenario_id,family_key,split,actual_fault_cells,predicted_fault_cells,false_positive_cells,false_negative_cells,actual_fault_count,predicted_fault_count,false_positive_count,false_negative_count,exact_cell_set_match,healthy_false_alarm
0,S3_HEALTHY,S3,SCENARIO::S3::FAULTS::,test,[],[],[],[],0,0,0,0,True,False
1,S4_CELL019,S4,SCENARIO::S4::FAULTS::19,test,[19],[19],[],[],1,1,0,0,True,False
2,S5_CELL091,S5,SCENARIO::S5::FAULTS::91,test,[91],[91],[],[],1,1,0,0,True,False
3,S6_S5_CELL091_0p4C,S6,SCENARIO::S5::FAULTS::91,test,[91],[91],[],[],1,1,0,0,True,False
4,S6_S5_CELL091_0p8C,S6,SCENARIO::S5::FAULTS::91,test,[91],[91],[],[],1,1,0,0,True,False
5,S6_S5_CELL091_1p0C,S6,SCENARIO::S5::FAULTS::91,test,[91],[91],[],[],1,1,0,0,True,False
6,S6_S5_CELL091_1p5C,S6,SCENARIO::S5::FAULTS::91,test,[91],[91],[],[],1,1,0,0,True,False
7,S6_S5_CELL091_2p0C,S6,SCENARIO::S5::FAULTS::91,test,[91],[91],[],[],1,1,0,0,True,False


In [12]:
#save the scenario-level results
scenario_out = RESULTS_DIR / "08_family_heldout_scenario_results.csv"
scenario_df.to_csv(scenario_out, index=False)

print("Saved scenario results to:", scenario_out)


Saved scenario results to: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/08_family_heldout_scenario_results.csv


In [17]:

# Cell 11 — Research checks and limitations

families = metadata[["family_key", "evaluation_split"]].drop_duplicates()

split_sets = {
    s: set(
        families.loc[families["evaluation_split"] == s, "family_key"]
    )
    for s in ["train", "val", "test"]
}

test_meta = metadata[metadata["evaluation_split"] == "test"]
train_rows = cell_eval[cell_eval["evaluation_split"] == "train"]

# Check that each scenario has exactly 100 distinct cells
cells_per_scenario = (
    cell_eval.groupby("scenario_instance_id")["cell_id"].nunique()
)
coverage_ok = bool(
    len(cells_per_scenario) > 0
    and (cells_per_scenario == 100).all()
)

checks = {
    "unique_scenario_metadata_ids": bool(metadata[ID].is_unique),

    "all_score_scenarios_have_metadata": bool(
        set(cell_eval["scenario_instance_id"]).issubset(set(metadata[ID]))
    ),

    "all_scored_scenarios_have_100_cells": coverage_ok,

    "no_train_test_family_overlap": bool(
        not (split_sets["train"] & split_sets["test"])
    ),

    "no_train_validation_family_overlap": bool(
        not (split_sets["train"] & split_sets["val"])
    ),

    "no_validation_test_family_overlap": bool(
        not (split_sets["val"] & split_sets["test"])
    ),

    "training_contains_healthy_cells": bool(
        (train_rows["is_fault"] == 0).any()
    ),

    "training_contains_faulty_cells": bool(
        (train_rows["is_fault"] == 1).any()
    ),

    "test_contains_healthy_scenarios": bool(
        (~test_meta["has_fault"]).any()
    ),

    "test_contains_fault_scenarios": bool(
        test_meta["has_fault"].any()
    ),

    "threshold_uses_train_only": True,
    "fault_onset_timestamps_available": False
}

print("RESEARCH CHECKS")
print(json.dumps(checks, indent=2))

# Report limitations
if not checks["all_scored_scenarios_have_100_cells"]:
    print("FAIL: Some scenarios do not contain exactly 100 distinct cells.")

if not checks["no_train_test_family_overlap"]:
    print("FAIL: Training and test share one or more fault families.")

if not checks["no_train_validation_family_overlap"]:
    print("FAIL: Training and validation share one or more fault families.")

if not checks["no_validation_test_family_overlap"]:
    print("FAIL: Validation and test share one or more fault families.")

if not checks["test_contains_healthy_scenarios"]:
    print("LIMITATION: Healthy held-out false-alarm performance cannot be evaluated.")

if not checks["test_contains_fault_scenarios"]:
    print("LIMITATION: Held-out fault detection cannot be evaluated.")

if not checks["fault_onset_timestamps_available"]:
    print("LIMITATION: Early-warning lead time cannot be calculated without fault-onset timestamps.")

print("Cell 11 completed.")


RESEARCH CHECKS
{
  "unique_scenario_metadata_ids": true,
  "all_score_scenarios_have_metadata": true,
  "all_scored_scenarios_have_100_cells": true,
  "no_train_test_family_overlap": true,
  "no_train_validation_family_overlap": true,
  "no_validation_test_family_overlap": true,
  "training_contains_healthy_cells": true,
  "training_contains_faulty_cells": true,
  "test_contains_healthy_scenarios": true,
  "test_contains_fault_scenarios": true,
  "threshold_uses_train_only": true,
  "fault_onset_timestamps_available": false
}
LIMITATION: Early-warning lead time cannot be calculated without fault-onset timestamps.
Cell 11 completed.


In [18]:
# Cell 12 — Save outputs and summary JSON
OUT_METRICS = RESULTS_DIR / "08_family_holdout_threshold_metrics.csv"
OUT_SCENARIOS = RESULTS_DIR / "08_family_holdout_scenario_details.csv"
OUT_SPLITS = RESULTS_DIR / "08_family_holdout_split_manifest.csv"
OUT_CELLS = RESULTS_DIR / "08_family_holdout_cell_predictions.csv"
OUT_SUMMARY = RESULTS_DIR / "08_family_holdout_threshold_summary.json"

metrics_df.to_csv(OUT_METRICS, index=False)
scenario_df.to_csv(OUT_SCENARIOS, index=False)
families.sort_values(["evaluation_split", "family_key"]).to_csv(OUT_SPLITS, index=False)
cell_eval.drop(columns=["fault_cells"]).to_csv(OUT_CELLS, index=False)

def json_safe_records(df):
    return json.loads(df.replace({np.nan: None}).to_json(orient="records"))

test_records = json_safe_records(metrics_df[metrics_df["split"] == "test"])
summary = {
    "notebook": "08_Family_HeldOut_Threshold_Calibration_Evaluation",
    "status": "COMPLETED_WITH_LIMITATIONS",
    "scenario_count": int(metadata[ID].nunique()),
    "family_count": int(metadata["family_key"].nunique()),
    "score_row_count": int(len(cell_eval)),
    "threshold_calibration": threshold_report,
    "split_source": split_source,
    "split_overview": json_safe_records(split_overview),
    "metrics_by_split": json_safe_records(metrics_df),
    "test_metrics": test_records[0] if test_records else None,
    "checks": checks,
    "limitations": [
        "Simulated/project dataset; not independent real-world vehicle validation.",
        "Generalization claims require genuinely distinct held-out fault families.",
        "Specificity is not established if test has no healthy scenarios.",
        "Early-warning lead time is not measured without fault-onset timestamps.",
        "This signal detector is separate from the PINN future-SOH localization method.",
        "No operational battery-safety threshold is validated by this notebook."
    ],
    "outputs": {
        "metrics_csv": str(OUT_METRICS),
        "scenario_details_csv": str(OUT_SCENARIOS),
        "family_split_manifest_csv": str(OUT_SPLITS),
        "cell_predictions_csv": str(OUT_CELLS),
        "summary_json": str(OUT_SUMMARY)
    }
}
with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Saved:")
for path in [OUT_METRICS, OUT_SCENARIOS, OUT_SPLITS, OUT_CELLS, OUT_SUMMARY]:
    print(path)
print("Threshold:", threshold)
print("Training score separation:", train_separated)
print("Run completed; review all limitation flags before reporting results.")


Saved:
/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/08_family_holdout_threshold_metrics.csv
/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/08_family_holdout_scenario_details.csv
/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/08_family_holdout_split_manifest.csv
/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/08_family_holdout_cell_predictions.csv
/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/08_family_holdout_threshold_summary.json
Threshold: 4.399034460052955
Training score separation: True
Run completed; review all limitation flags before reporting results.


## Interpretation rule

Do not claim generalization from the previous 100% result unless the test contains entirely held-out fault families. A healthy test scenario is required to measure false alarms/specificity. This notebook does not compute warning lead time and does not validate real-world safety thresholds.
